# Task 1

## 1. IoU entre cajas

Cajas:

$$
{b} = (142, 89, 218, 165), \quad b^* = (138, 84, 222, 170)
$$

Primero sacamos la intersección (la parte donde ambas cajas se traslapan):

$$
x_{min} = \max(142,138)=142,\quad x_{max}=\min(218,222)=218
$$

$$
y_{min} = \max(89,84)=89,\quad y_{max}=\min(165,170)=165
$$

Esto nos da un cuadro común de:

$$
76 \times 76 \Rightarrow |I| = 5776
$$

Luego áreas individuales (para saber cuánto cubre cada caja):

$$
A_{pred}=5776,\quad A_{gt}=84 \times 86 = 7224
$$

La unión combina ambas áreas sin duplicar la intersección:

$$
|U| = 5776 + 7224 - 5776 = 7224
$$

Finalmente:

$$
IoU = \frac{5776}{7224} \approx 0.80
$$

### ¿Qué significa esto?

Un **IoU de 0.80** es bastante bueno: la detección está bien alineada con el producto real.  
por lo tanto le explicaría: En que no hay problema con que los numeros no parezcan matchear desde el inicio ya que tenemos un alto "encaje" del producto real con la detección por lo tanto funcionará.

---

## 2. Interpretación de la fórmula

$$
IoU = \frac{|I|}{|U|}
$$

$$
|I|
$$

: lo que ambas cajas comparten

$$
|U|
$$

: todo lo que cubren juntas

--- 

Si usáramos el ground truth:

$$
\frac{|I|}{|GT|}
$$

Con una caja enorme que cubra todos los productos del shelf seria suficiente para tener score perfecto porque si o sí estaría dentro el producto. 

Usar la unión penaliza eso automáticamente:
- cajas muy grandes → unión crece → IoU baja  
- cajas mal alineadas → menos overlap → IoU baja  


---

## 3. Elección de umbral IoU

Opciones:

$$
\theta = 0.5
$$

(relajado)
$$
\theta = 0.75
$$

(estricto)


Se prioriza el falso positivo leve que un falso negativo ya que no poder identificar un producto que si esta se empieza a perder el inventario generando problemas mas costosos que detectar una posible caja imperfecta por lo tanto el balance entre estricto y suficiente es necesario para evitar perder objetos.

---

### Recomendación final

$$
\theta = 0.5
$$

### ¿Por qué?

- Prioriza **recall** 
- Evita alertas falsas de stock vacío

En resumen:
Mejor detectar “bien suficiente” que fallar detecciones críticas.



## Pregunta 1.2
## 4. Precisión y Recall

Datos:

$$
TP = 12,\quad FP = 6,\quad FN = 3
$$

---

### Precisión

$$
P = \frac{TP}{TP + FP} = \frac{12}{12+6} = \frac{12}{18} \approx 0.67
$$

Aquí el $(TP + FP)$ son las detecciones que el modelo hizo.  
La precisión seria de cuales de las predicciones estas son correctas

---

### Recall

$$
R = \frac{TP}{TP + FN} = \frac{12}{12+3} = \frac{12}{15} = 0.80
$$

Aquí el denominador $(TP + FN)$ son  los productos reales en el anaquel  
El recall seria de lo que existía cuanto se logró detectar?

---

- Precisión ≈ **0.67** → hay ruido (algunas detecciones falsas)  
- Recall = **0.80** → detecta la mayoría de productos  

---

### ¿Por qué necesitamos ambas?

Porque miden cosas distintas:

- Precisión → calidad de las detecciones  
- Recall → cobertura del sistema  

---

## 5. Preferencia del negocio → métricas


> "Prefiero no perder quiebres de stock"

Lo cual su traducción técnica sería:

Quiere **alto recall**  
Acepta bajar precisión

---


### Ajuste práctico

Bajar el **threshold de confianza**

- Más bajo → más detecciones  
- ↑ TP pero también ↑ FP  
- ↓ FN  

---

## 6. ¿Qué es mAP?

mAP = **mean Average Precision**

No es una sola precisión, sino un promedio de desempeño considerando los dif. thresholds.

En vez de medir en un solo punto (ej: threshold fijo),  
mAP evalúa cómo se comporta el modelo en todo el rango de decisiones.

Es como ver toda la curva Precision-Recall, no solo un punto.

---

### Diferencia de protocolos

**mAP@0.5 (PASCAL VOC):**

- Solo evalúa con IoU ≥ 0.5  
- Más permisivo  
- Más fácil obtener buen score  

---

**mAP@0.5:0.95 (COCO):**

- Promedia IoU desde 0.5 hasta 0.95  
- Incluye localización muy **precisa**  
- Mucho más estricto  

---

### ¿Cuál es más exigente?

**COCO (0.5:0.95)**

Porque no solo pide detectar el objeto, sino ubicarlo muy bien.

- mAP@0.5 con suficiente para saber si hay producto  
- mAP@0.5:0.95 para evaluar qué tan bien está alineada la caja  

COCO es más exigente porque penaliza errores pequeños de localización  
(que en anaqueles pueden afectar conteo o layout)



## Pregunta 1.3
## 7. ¿Qué es NMS y por qué hay tantas cajas?

Lo que está pasando es normal en detección de objetos.

El modelo no “ve una caja por producto”, sino que prueba muchas posibles ubicaciones (anchors o candidatos) y varias coinciden sobre el mismo objeto.

NMS(Non Maximum Suppression), es un filtro que se queda solo con **la mejor caja por objeto** y elimina duplicados.

### Algoritmo

1. El modelo asigna un **score de confianza** a cada caja  
2. Se ordenan las cajas de mayor a menor score  
3. Se toma la mejor caja (la más confiable)  
4. Se eliminan todas las cajas que se traslapan mucho con ella (IoU alto)  
5. Se repite con las cajas restantes  

Resultado: una caja por producto (o pocas), en lugar de decenas.

---

## 8. Elección de θ_NMS en anaquel denso

En anaqueles:

- productos están muy juntos, normalmente tocandose  
- cajas legítimas pueden traslaparse un poco  

---

**Si θ_NMS es muy bajo (agresivo)**

- elimina muchas cajas  
- riesgo: borrar productos reales que están pegados  

mas falsos negativos  

**Si θ_NMS es alto (menos agresivo)**

- permite más overlap  
- mantiene cajas cercanas  

menos riesgo de perder productos pero mas duplicados (FP)

---

### Recomendación

Nuevamente se trata de balance, usar **θ_NMS alto (ej: 0.5 – 0.7)**

Porque:

- mejor tolerar duplicados  
- que eliminar productos reales  

En retail:

perder un producto (FN) es peor que duplicarlo (FP leve)

---

## 9. Orden: threshold vs NMS

El orden correcto es:

1. Aplicar **threshold de confianza**
2. Luego aplicar **NMS**

---

### ¿Por qué?

Primero se elimina cajas con baja confianza o sea el ruido:

- se reduce cantidad de cajas    

Luego NMS trabaja sobre un conjunto más limpio

Poruqe si se hace NMS primero:

- NMS procesa **todas las 312 cajas**  
- muchas son basura (bajo score)  
Se hace trabajo doble por esos datos basura

**En el contexto:** 

Sistema procesa:

- 30 imágenes por minuto  
- sin GPU  

Si no filtras primero:

- aumenta latencia  
- puedes pasar de <500ms a tiempos no aceptables  



## **Task 2**

## Pregunta 2.1

### 1. Cuello de botella principal de R-CNN

El cuello de botella principal de **R-CNN** está en que procesa cada propuesta de región de manera **independiente**. Si una imagen genera alrededor de **2,000 regiones**, la red debe ejecutar casi **2,000 pasadas completas por la CNN**, una por cada recorte. Eso vuelve el sistema extremadamente costoso en tiempo y en cómputo.

El problema no es solo la cantidad de regiones, sino lo que ocurre dentro de cada pasada. En cada una, la red vuelve a aplicar convoluciones, activaciones, pooling y capas totalmente conectadas sobre una región que muchas veces se superpone con otras. En otras palabras, la red recalcula una y otra vez información visual muy parecida. Esa redundancia hace que el costo crezca de forma innecesaria. Para un entorno como VisorShelf, donde el hardware de tienda es limitado, ese diseño no resulta operativo.

### 2. Cómo Fast R-CNN resuelve ese cuello de botella

**Fast R-CNN** introduce dos ideas clave: el **feature map compartido** y el **RoI Pooling**.

El **feature map compartido** elimina la necesidad de correr la CNN por cada propuesta. En lugar de recortar 2,000 regiones y procesarlas una por una, primero se pasa **la imagen completa** por la red convolucional una sola vez. El resultado es un mapa de características profundo que concentra la información visual relevante de toda la imagen. Luego, cada región propuesta se proyecta sobre ese mapa ya calculado. Con eso se elimina el cómputo más caro y repetitivo de R-CNN: volver a ejecutar las convoluciones para cada propuesta.

El **RoI Pooling** resuelve otro problema: las regiones propuestas no tienen el mismo tamaño, pero las capas posteriores de la red necesitan una entrada de tamaño fijo. Para resolverlo, el RoI Pooling toma cada región proyectada sobre el feature map, la divide en una cuadrícula fija, por ejemplo de $$ H \times W $$, y dentro de cada celda aplica una operación de **max pooling**. Así convierte una región de tamaño variable en un tensor de tamaño fijo.

La lógica es simple: el feature map compartido evita repetir convoluciones, y el RoI Pooling permite estandarizar regiones irregulares para que puedan entrar a las capas finales del detector.

### 3. Nuevo cuello de botella en Fast R-CNN

Aunque **Fast R-CNN** redujo drásticamente el tiempo de la CNN, el tiempo total seguía siendo alto porque el nuevo cuello de botella pasó a ser **Selective Search**, el algoritmo usado para generar propuestas de región.

El problema de **Selective Search** no es únicamente que sea lento. El problema más profundo es que está **fuera de la red** y no forma parte de un sistema entrenable de extremo a extremo. Es un módulo externo, heurístico y artesanal, que no aprende a proponer regiones según el dataset específico de VisorShelf. Eso significa que el detector depende de un generador de regiones que no se optimiza junto con la clasificación ni con la regresión de cajas.

Desde una perspectiva arquitectónica, eso introduce varias limitaciones: rompe la integración del pipeline, dificulta aprovechar aceleración conjunta, complica la optimización global del sistema y hace que la calidad de las propuestas no evolucione con el entrenamiento del detector. Por eso, el problema de Selective Search no es solo de velocidad, sino de diseño de sistema.

---

## Pregunta 2.2

### 4. Por qué sí se necesita una Region Proposal Network

La pregunta del ingeniero junior es razonable, pero un **sliding window clásico** sobre el feature map no resuelve el problema de la misma manera que una **Region Proposal Network (RPN)**.

La **RPN** no se limita a recorrer posiciones; en cada posición del feature map aprende a responder dos preguntas: **si allí hay o no un objeto** y **cómo debe ajustarse una caja candidata para aproximarse mejor al objeto real**. Es decir, no solo explora el espacio, sino que también **puntúa** y **refina** hipótesis de detección.

Un sliding window clásico, en cambio, recorre ventanas predefinidas de forma rígida. Puede cubrir el espacio, pero no tiene por sí mismo un mecanismo tan eficiente y entrenable para aprender cuáles regiones merecen atención y cómo corregirlas geométricamente. Termina siendo más tosco, menos adaptativo y más costoso en cantidad de ventanas evaluadas.

Además, que la **RPN opere sobre el mismo feature map del backbone** no es solo una ventaja computacional, sino también **semántica**. Significa que las propuestas se generan a partir de representaciones profundas que ya contienen información de bordes, texturas, formas y patrones de objeto. En otras palabras, la red no propone regiones solo por geometría, sino con base en una comprensión visual más rica. Eso hace que las propuestas estén mejor alineadas con el detector final.

### 5. Qué son los anchors y por qué se predicen deltas

Los **anchors** son cajas de referencia predefinidas que se colocan en cada posición del feature map. En Faster R-CNN, normalmente se usan varias por posición combinando distintas **escalas** y **relaciones de aspecto**. Por eso se habla, por ejemplo, de **9 anchors por posición**: 3 escalas por 3 proporciones.

La idea es que, en lugar de pedirle a la red que invente una caja desde cero, se le da un conjunto de hipótesis iniciales razonables. Luego, la red aprende a ajustar cada anchor prediciendo desplazamientos y cambios de tamaño.

Por eso la red predice **deltas** $$(\Delta x, \Delta y, \Delta w, \Delta h)$$ en lugar de coordenadas absolutas. Es más fácil aprender una **corrección relativa** respecto a una caja base que aprender directamente coordenadas globales desde cero. Ese enfoque estabiliza el entrenamiento y vuelve la regresión más consistente entre objetos de tamaños distintos.

En la ecuación:

$$
w = w_a \cdot e^{\Delta w}
$$

-  w representa el **ancho final** de la caja predicha.
-  w_a representa el **ancho del anchor**.
- Delta w  representa el **ajuste aprendido por la red** para esa dimensión.

Se usa la exponencial porque el ancho y la altura deben ser siempre **positivos**, y además porque el cambio de escala se modela mejor de forma **multiplicativa** que aditiva. Un objeto puede ser el doble o la mitad de ancho que el anchor; la exponencial permite representar ese tipo de relación de forma natural.

### 6. Recomendación de Faster R-CNN para producción en hardware de tienda

Para producción en el hardware de tienda, **no se recomendaría Faster R-CNN como solución principal**, al menos no en la configuración planteada y sin GPU.

Aunque la referencia de aproximadamente **5 FPS con VGG16** parece suficiente frente a una meta de **más de 2 FPS**, ese valor no debe interpretarse fuera de contexto. En la práctica, ese rendimiento suele asociarse a hardware mucho más favorable que una CPU de tienda. En CPU, el desempeño real probablemente caería por debajo de la meta operativa.

Más allá de la velocidad, hay al menos dos factores adicionales que deben considerarse. El primero es que **Faster R-CNN sí tiene una ventaja importante en precisión**, especialmente cuando hay objetos pequeños, cercanos entre sí o parcialmente ocluidos, algo muy relevante en anaqueles densos. El segundo es que tiene un **ecosistema de implementación maduro**, con bastante soporte en frameworks y literatura, lo cual facilita el fine-tuning y la depuración.

Sin embargo, esas ventajas no compensan del todo el costo operativo en una tienda sin GPU. También hay que considerar el **tamaño del modelo**, el consumo de memoria, la mayor complejidad del pipeline y la variabilidad que puede introducir en un entorno productivo con recursos limitados. Por ello, Faster R-CNN podría servir como línea base de alta precisión o como sistema para evaluación offline, pero no como primera opción de despliegue en CPU de tienda.

---

## Pregunta 2.3

### 7. Recomendación entre Propuesta A y Propuesta B

Tomando en cuenta las restricciones reales de VisorShelf —**sin GPU, latencia menor a 500 ms y anaqueles densos**— la recomendación más razonable sería la **Propuesta B: YOLOv8n fine-tuned**.

La razón principal es que, en este caso, la restricción de latencia no es negociable. Si el sistema debe responder por debajo de 500 ms en hardware de tienda, entonces no basta con que un modelo sea más preciso: también debe ser viable en operación diaria. La Propuesta A, aunque técnicamente más fuerte para objetos pequeños y densos gracias a **FPN**, se queda corta en CPU. Su velocidad estimada de alrededor de **1.5 FPS** implica un tiempo cercano o superior a lo permitido, mientras que **YOLOv8n**, con alrededor de **15 FPS en CPU**, sí entra con margen suficiente.

Además, la Propuesta B tiene ventajas operativas claras. Su tamaño es mucho menor, el despliegue es más liviano, el consumo de recursos es menor y el fine-tuning es más sencillo. Todo eso importa en un sistema que debe mantenerse en varias tiendas, no solo en un entorno de laboratorio.

Ahora bien, no debe ignorarse que la Propuesta A tiene una ventaja real en escenas de anaqueles densos, porque la combinación de arquitectura de dos etapas y FPN suele manejar mejor objetos pequeños y cercanos. Sin embargo, en este contexto la mejor decisión no es la que maximiza el mAP en abstracto, sino la que mantiene un equilibrio entre precisión y viabilidad. Un detector que identifica mejor pero no cumple la latencia operativa deja de ser una buena decisión de producto.

Por eso, para producción en tiendas sin GPU, la opción más sensata sería **YOLOv8n**, idealmente acompañada de estrategias para recuperar parte de la precisión en objetos pequeños, como aumento de resolución, recortes por secciones del anaquel o augmentations específicas del dominio.

### 8. Si hubiera una GPU RTX 3060 en tiendas flagship

Sí, con una **RTX 3060** la recomendación podría cambiar, al menos para las tiendas flagship.

Con GPU, la Propuesta A deja de estar bloqueada por la latencia y se vuelve una opción mucho más atractiva. Si logra entre **8 y 12 FPS**, entonces ya cumple sobradamente el requisito temporal. En ese nuevo escenario, el trade-off cambia: la velocidad deja de ser la barrera principal y pasa a pesar más la **calidad de detección**, especialmente en objetos pequeños, densos y con poca separación visual.

Ahí la arquitectura **Faster R-CNN + ResNet-50 + FPN** gana valor, porque su diseño multi-escala y su naturaleza de dos etapas pueden ofrecer mejores resultados en anaqueles complejos. En una tienda flagship, donde probablemente se espera un rendimiento más robusto y donde el costo de hardware puede justificarse mejor, esa mejora en precisión puede ser estratégica.

Aun así, no necesariamente habría que reemplazar por completo la Propuesta B en todo el sistema. Podría existir una estrategia mixta: **YOLOv8n** para tiendas estándar con CPU y **Faster R-CNN + FPN** para tiendas flagship con GPU. Ese enfoque sería coherente con una arquitectura escalonada según capacidades de hardware.

### 9. Riesgo de fine-tuning sin learning rate diferenciado

Hacer fine-tuning de **Faster R-CNN** en un dataset de anaqueles sin aplicar un **learning rate diferenciado** entre el backbone preentrenado y las capas nuevas introduce un riesgo claro: que el backbone pierda demasiado rápido las representaciones útiles aprendidas durante el preentrenamiento.

Ese fenómeno se conoce comúnmente como **catastrophic forgetting** o **olvido catastrófico**. Ocurre cuando los pesos de una red preentrenada se actualizan con demasiada agresividad en una nueva tarea y terminan destruyendo características generales valiosas, como bordes, texturas o patrones visuales útiles. En este caso, el riesgo aumenta porque las capas nuevas suelen iniciar con pesos aleatorios y generan gradientes más inestables; si el backbone recibe el mismo learning rate, puede degradarse antes de adaptarse correctamente al dominio de los anaqueles.

La mitigación más recomendada consiste en usar un **learning rate menor para el backbone** y uno más alto para las capas nuevas. También puede congelarse inicialmente parte del backbone y luego ir liberándolo de forma gradual. A eso se le puede sumar **warmup**, monitoreo cuidadoso de validación y, si es necesario, un esquema de entrenamiento por etapas. Con eso se protege el conocimiento previo del modelo mientras las nuevas capas aprenden la tarea específica de VisorShelf.